# RT-DETR PPE fine-tuning -- Colab runner

**Run every cell below, IN ORDER, top to bottom. Don't skip any.**

Before starting: Runtime menu (top left) -> Change runtime type -> Hardware accelerator: T4 GPU -> Save.

You will be prompted partway through to upload a file called `data_raw.zip` --
prepare it BEFORE you start by running this on your own Windows machine:
```
cd C:\Users\admin\Downloads\RAP\ppe-detection-api
Compress-Archive -Path data\raw -DestinationPath data_raw.zip
```
That creates `data_raw.zip` inside your `ppe-detection-api` folder -- have it ready to pick when Colab asks.

## Cell 1 -- Install dependencies and confirm GPU

In [ ]:
!pip install -q ultralytics
!nvidia-smi  # record this output -- goes in your memo's reproducibility section

## Cell 2 -- Clone your GitHub repo (gets you src/, api/, data/data.yaml)

In [ ]:
!git clone https://github.com/Bhuvana-Manogar/Constrained-Object-Detection-Reasoning-API.git repo
%cd repo

## Cell 3 -- Upload your dataset zip
Running this cell opens a file-picker button. Click it and select `data_raw.zip` from your computer.

In [ ]:
from google.colab import files
uploaded = files.upload()

## Cell 4 -- Unzip the dataset into the right place

In [ ]:
!mkdir -p data/raw
!unzip -q data_raw.zip -d data/raw_tmp
# Compress-Archive on Windows preserves the 'raw' folder itself inside the zip,
# so the real content lands one level deeper -- this moves it up to data/raw.
!mv data/raw_tmp/raw/* data/raw/ 2>/dev/null || mv data/raw_tmp/* data/raw/
!ls data/raw

## Cell 5 -- Verify the dataset landed correctly
You should see two numbers, both around 2801 (or whatever your local count showed).

In [ ]:
import os
print('images:', len(os.listdir('data/raw/images')))
print('labels:', len(os.listdir('data/raw/labels')))

## Cell 6 -- Split (same seed as your local run, so results match)

In [ ]:
!python src/split_dataset.py --source data/raw --dest data/split --seed 42

## Cell 7 -- Train (this is the slow one -- can take 30-90+ minutes depending on epochs)

In [ ]:
!python src/train.py --data data/data.yaml --epochs 100 --imgsz 640 --batch 16 --device 0

## Cell 8 -- Evaluate on the held-out test split

In [ ]:
!python src/evaluate.py --weights runs/ppe_rtdetr/weights/best.pt --data data/data.yaml --split test

## Cell 9 -- Download your trained weights back to your computer
This downloads `best.pt` -- move it into your local repo's `models/best.pt` afterward.

In [ ]:
from google.colab import files
files.download('runs/ppe_rtdetr/weights/best.pt')

## Cell 10 -- Also download the confusion matrix + PR curve images for your memo's failure-case section

In [ ]:
import glob
from google.colab import files
for f in glob.glob('runs/detect/val*/confusion_matrix.png') + glob.glob('runs/detect/val*/*curve*.png'):
    print('downloading', f)
    files.download(f)